In [1]:
import pandas as pd

df = pd.read_csv('../dataset/processed/spam_cleaned.csv')

## Cross Validation

In [2]:
# Now we use TfidVectorizer
from sklearn.naive_bayes import GaussianNB,MultinomialNB,BernoulliNB
from sklearn.model_selection import cross_val_score,cross_validate
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer 
tfidf =TfidfVectorizer()
count =CountVectorizer()

In [3]:
# Clean the data: drop rows with NaN in 'transformed_text'
df = df.dropna(subset=['transformed_text'])
print("After dropping NaN, total rows:", len(df))

After dropping NaN, total rows: 5160


In [4]:
# create a pipeline
pipe =Pipeline([
    ('tfidf',TfidfVectorizer(max_features=3000)),
    ('model',MultinomialNB())
])

# metrics
score={
    'accu':'accuracy',
    'preci':'precision'
}
# Now cross_validate mean()
cv_result= cross_validate(
    pipe,
    df['transformed_text'], # tfidf fitted sperately inside each fold
    df['target'],# no data leakage occur 
    cv=5,
    scoring=score
)

#  cross_validate mean()

#Accuracy calculated on the validation fold(the test part of that fold)
print(f'Accuracy:,{cv_result['test_accu'].mean()}') 
print(f'Precision:,{cv_result['test_preci'].mean()}')

Accuracy:,0.9686046511627906
Precision:,0.9916666666666666


In [5]:
# Check for NaN values in 'transformed_text'
print("Number of NaN values in 'transformed_text':", df['transformed_text'].isnull().sum())
print("Total rows:", len(df))
print("Sample of 'transformed_text':", df['transformed_text'].head())

Number of NaN values in 'transformed_text': 0
Total rows: 5160
Sample of 'transformed_text': 0    go jurong point crazi avail bugi n great world...
1                                ok lar joke wif u oni
2    free entri 2 wkli comp win fa cup final tkt 21...
3                  u dun say earli hor u c alreadi say
4                 nah think goe usf live around though
Name: transformed_text, dtype: object


In [6]:
#Now when we doing multiple ml model and different metics 
from sklearn.preprocessing import FunctionTransformer

#ml models
cross_val_model={
    'gnb': GaussianNB(),
    'mnb':MultinomialNB(),
    'bnb':BernoulliNB()
}

# metrics
score1 ={
    'accu':'accuracy',
    'prec':'precision',
}

# vectorizer
vectorizers ={
    'TF-IDF':TfidfVectorizer(max_features=3000),
    'Count':CountVectorizer(max_features=3000)
}

# loop 
for vect_name,vect in vectorizers.items():
    for name, model in cross_val_model.items():
        if isinstance(model,GaussianNB):
         pipe1 =Pipeline([
             ('tfidf',vect),
             ('to_dense',FunctionTransformer(lambda x:x.toarray(),accept_sparse=True)),
             ('model',model) 
         ])
        else:
         pipe1 =Pipeline([
             ('vectorizer',vect),
             ('model',model)
          ])
                
        cv_result1=cross_validate(
            pipe1,
            df['transformed_text'], # tfidf fitted sperately inside each fold
            df['target'],# no data leakage occur 
            cv=5,
            scoring=score1
        )

        print(f'\nVectorizer:{vect_name},Model:{name} Results:')
        print('Accuracy :', cv_result1['test_accu'].mean())
        print('Precision:', cv_result1['test_prec'].mean())

        


Vectorizer:TF-IDF,Model:gnb Results:
Accuracy : 0.8443798449612403
Precision: 0.43958463333505665

Vectorizer:TF-IDF,Model:mnb Results:
Accuracy : 0.9686046511627906
Precision: 0.9916666666666666

Vectorizer:TF-IDF,Model:bnb Results:
Accuracy : 0.9765503875968993
Precision: 0.9908542046009996

Vectorizer:Count,Model:gnb Results:
Accuracy : 0.8474806201550388
Precision: 0.4470949607108416

Vectorizer:Count,Model:mnb Results:
Accuracy : 0.9782945736434108
Precision: 0.9365551294738852

Vectorizer:Count,Model:bnb Results:
Accuracy : 0.9765503875968993
Precision: 0.9908542046009996


In [7]:
# FOR MY NEED I CHOOSE MultinomialNB + TF-IDF I.E 
# .toarray() only used for GaussianNB
from sklearn.model_selection import train_test_split

X = df['transformed_text']
y = df['target']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

mnb_pipeline=Pipeline([
    ('tfidf',TfidfVectorizer()),
    ('mnb',MultinomialNB())
])

#fit and predict
mnb_pipeline.fit(X_train,y_train)
y_pred=mnb_pipeline.predict(X_test)

# accuracy and precission score
print(f'accuracy:{accuracy_score(y_test,y_pred)}')
print(f'precission:{precision_score(y_test,y_pred)}')
print(f'confusion_matrix:{confusion_matrix(y_test,y_pred)}')

accuracy:0.9618863049095607
precission:1.0
confusion_matrix:[[1363    0]
 [  59  126]]


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier

# fetaure and target
X = df['transformed_text']
y = df['target']

# split train test 
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

# ml models
models={
    'mnb':MultinomialNB(),
    'lr':LogisticRegression(max_iter=500),
    'svm':SVC(),
    'rf':RandomForestClassifier(n_estimators=100),
    'gb':GradientBoostingClassifier(),
}

# list
results=[]

#loop through model
for model_name,model in models.items():
    pipeline=Pipeline([
        ('tfidf',TfidfVectorizer(max_features=3000)),
        ('model',model)
    ])

    # fit on trainind data
    pipeline.fit(X_train,y_train)
    y_pred=pipeline.predict(X_test)
    acc = accuracy_score(y_test,y_pred)
    prec= precision_score(y_test,y_pred,average='macro')

    #append the result
    results.append({
        'model':model_name,
        'accuracy':acc,
        'precision':prec
    })

# convert into dataframe
result_df =pd.DataFrame(results)

# sort the data
result_df=result_df.sort_values(by='accuracy',ascending=False).reset_index(drop=True)

print(result_df)

  model  accuracy  precision
0   svm  0.978036   0.979303
1    rf  0.974160   0.968670
2   mnb  0.968992   0.982991
3    gb  0.965116   0.946215
4    lr  0.962532   0.959964


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score

# Base models
svm = SVC(probability=True)
mnb = MultinomialNB()
rf = RandomForestClassifier()

# Stacking classifier
estimators = [
    ('svm', svm),
    ('mnb', mnb),
    ('rf', rf)
]

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    passthrough=True
)

# Pipeline: vectorize text first
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=3000)),
    ('stack', stacking_clf)
])

# Fit pipeline
pipeline.fit(X_train, y_train)

# Predict
y_pred = pipeline.predict(X_test)

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='macro'))


Accuracy: 0.9799741602067183
Precision: 0.965843023255814
